# Conversation-Aware Chatbot with Gradio

In this notebook, I build a chatbot that maintains conversation history, allowing the model to generate responses using the context of previous messages instead of treating every prompt as a separate request.

I use Gradio to create an interactive chat interface and the OpenAI API to power the chatbot. Through this project, I demonstrate how a stateless LLM can be used to create a stateful chat experience by storing and sending conversation history with each request.

Key concepts: chatbot UI, conversation history, message roles, stateless vs. stateful LLM behavior, Gradio ChatInterface, OpenAI API integration, and context-aware conversations.

In [1]:
import os  # I use this to read environment variables from my computer.

from dotenv import load_dotenv  # I use this to load secret keys from a .env file.

from openai import OpenAI  # I use this to create an OpenAI client so my code can call the OpenAI API.

import gradio as gr  # I use this to build a simple web app interface for my AI project.

In [2]:
# Loading my API key from the .env file, and checking if it exists, then print the first 8 characters to confirm it was found without exposing the full key.

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

OpenAI API Key exists and begins sk-proj-


In [3]:
# I create an OpenAI client so my code can send requests to OpenAI, then I choose which model I want to use.

openai = OpenAI()
MODEL = 'gpt-4.1-mini'

In [4]:

system_message = "You are a helpful assistant"

In [5]:
#callback function we are giving  gradio
#testing the gradio chatbot wiring works before connecting it to gpt 
def chat(message, history):
    return "bananas"

In [6]:
gr.ChatInterface(
    fn=chat,            # Callback function Gradio runs whenever the user sends a message
    type="messages"     # Sends and receives conversation as chat messages (user/assistant)
).launch()              # Starts the chatbot web application

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [8]:
def chat(message, history):
    return f"You said {message} and the history is {history} but I still say bananas"

In [9]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [10]:

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content


def chat(message, history):
    # Creates a callback function for Gradio ChatInterface.
    # message = the latest message the user typed.
    # history = the previous conversation stored by Gradio.

    history = [{"role": h["role"], "content": h["content"]} for h in history]
    # Converts Gradio's history into the message format OpenAI expects.
    # Each message must look like:
    # {"role": "user", "content": "Hello"}
    # {"role": "assistant", "content": "Hi"}
    #
    # This is a list comprehension.
    # It loops through every message in history and keeps only role + content.

    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    # Builds the full conversation that will be sent to the model.
    #
    # It combines:
    # 1. system message = instruction for how the assistant should behave
    # 2. history = previous user/assistant messages
    # 3. current user message = the new message just typed
    #
    # This is what makes the chatbot conversation-aware.

    response = openai.chat.completions.create(model=MODEL, messages=messages)
    # Sends the full messages list to the OpenAI model.
    # MODEL is probably something like "gpt-4.1-mini".
    # The model now sees the system instruction, past conversation, and latest message.

    return response.choices[0].message.content
    # Extracts the assistant's answer from the API response.
    # Returns it back to Gradio so Gradio can display it in the chat UI.

In [11]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [12]:
system_message = "You are a helpful assistant in a clothes store. You should try to gently encourage \
the customer to try items that are on sale. Hats are 60% off, and most other items are 50% off. \
For example, if the customer says 'I'm looking to buy a hat', \
you could reply something like, 'Wonderful - we have lots of hats - including several that are part of our sales event.'\
Encourage the customer to buy hats if they are unsure what to get."

In [13]:
gr.ChatInterface(fn=chat, type="messages").launch()


* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
